In [1]:
# Parameters
frequency = "1d"


In [2]:
import numpy as np
import pandas as pd
from pylab import plt, mpl
from sklearn.metrics import accuracy_score
import os
import papermill as pm
import talib as ta
from sklearn.model_selection import TimeSeriesSplit

try:
    print(f"Frecuencia recibida desde papermill: {frequency}")
except NameError:
    print(f"No se recibió 'frequency'.")


# Cargar los datos para esta frecuencia
file_name = f"processed_data_{frequency}_charac.csv"
data = pd.read_csv(file_name, index_col='timestamp')
data


Frecuencia recibida desde papermill: 1d


,BTCUSDT_1d,LTCBTC_1d,BNBUSDT_1d,BNBBTC_1d,LTCUSDT_1d,ADABTC_1d,ADAUSDT_1d,SOLUSDT_1d,SOLBTC_1d
timestamp,,,,,,,,,
2020-08-11,11392.08,0.004751,21.2902,0.001869,54.10,0.000012,0.13685,3.2985,0.000289
2020-08-12,11564.33,0.004718,21.4919,0.001860,54.52,0.000012,0.13670,3.7558,0.000324
2020-08-13,11780.00,0.004850,21.7667,0.001849,57.14,0.000012,0.13944,3.7300,0.000316
2020-08-14,11760.54,0.004828,23.1047,0.001964,56.78,0.000012,0.13826,3.4099,0.000290
2020-08-15,11852.40,0.005050,23.0881,0.001949,59.85,0.000012,0.13837,3.1730,0.000268
...,...,...,...,...,...,...,...,...,...
2024-12-28,95300.00,0.001057,722.1300,0.007577,100.74,0.000009,0.88950,195.5000,0.002051
2024-12-29,93738.20,0.001051,694.7100,0.007412,98.51,0.000009,0.85900,189.9400,0.002027
2024-12-30,92792.05,0.001072,705.3600,0.007600,99.40,0.000009,0.86150,191.3800,0.002062


Función para guardar los datos. Hace un archivo por cada frecuencia. Guarda en cada línea el modelo que se ha empleado, el activo, accuracy e in/out-sample.

In [3]:
def save_results(model, ric, acc, sample, frequency=frequency):
    # Verificar si el archivo ya existe
    file_name = f'accuracy_results_{frequency}_charac.csv'

    # Si el archivo existe, leer los datos previos, si no, crear un nuevo DataFrame vacío
    if os.path.exists(file_name):
        df_results = pd.read_csv(file_name)
    else:
        df_results = pd.DataFrame(columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])

    # Agregar la nueva fila con los resultados
    new_row = pd.DataFrame([[model, ric, acc, sample]], columns=['Model', 'Asset', 'Accuracy', 'IN/OUT Sample'])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

    # Guardar los resultados acumulados
    df_results.to_csv(file_name, index=False) 

Creamos las características que usaremos para hacer el aprendizaje ahora y las retardamos.

In [4]:
def add_lags(data, ric, lags, window=30):
    cols = []
    df = pd.DataFrame(data[ric])
    df.dropna(inplace=True)
    df['r'] = np.log(df / df.shift()) #retornos
    df['sma'] = df[ric].rolling(window).mean()  #media movil de la ventana
    df['min'] = df[ric].rolling(window).min() #mínimo de la ventana
    df['max'] = df[ric].rolling(window).max() #máximo de la ventana
    df['mom'] = df['r'].rolling(window).mean() #momentum de la ventana
    df['vol'] = df['r'].rolling(window).std() #volatilidad de la ventana
    df['rsi'] = ta.RSI(df[ric], timeperiod=window) #rsi de la ventana
    df['atr'] = ta.ATR(df[ric], df[ric], df[ric], timeperiod=window) #atr de la ventana
    df.dropna(inplace=True)
    df['d'] = np.where(df['r'] > 0, 1, 0) # columna binaria, 0 si los precios bajarán, 1 si subirán
    features = [ric, 'r', 'd', 'sma', 'min', 'max', 'mom', 'vol', 'rsi', 'atr']
    for f in features:
        for lag in range(1, lags + 1):
            col = f'{f}_lag_{lag}'
            df[col] = df[f].shift(lag)
            cols.append(col)
    df.dropna(inplace=True)
    return df, cols

lags = 5

dfs = {}
for ric in data:
    df, cols = add_lags(data, ric, lags)
    dfs[ric] = df.dropna(), cols

Hacemos una función que entrene el modelo, lo valide utilizando walk-forward y calcule el accuracy.

In [5]:
from sklearn.neural_network import MLPClassifier

for ric in data:
    model = MLPClassifier(hidden_layer_sizes=[512],
                        random_state=100,
                        max_iter=1000,
                        early_stopping=True,
                        validation_fraction=0.15,
                        shuffle=False)
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'])
    pred = model.predict(df[cols])
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('MLPClassifier', ric, acc, "IN-SAMPLE")



IN-SAMPLE | BTCUSDT_1d | acc=0.5911


C:\Users\raque\AppData\Local\Temp\ipykernel_21300\3694455987.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_results = pd.concat([df_results, new_row], ignore_index=True)


IN-SAMPLE | LTCBTC_1d | acc=0.6529


IN-SAMPLE | BNBUSDT_1d | acc=0.5694


IN-SAMPLE | BNBBTC_1d | acc=0.6484


IN-SAMPLE | LTCUSDT_1d | acc=0.5624


IN-SAMPLE | ADABTC_1d | acc=0.6325


IN-SAMPLE | ADAUSDT_1d | acc=0.6771


IN-SAMPLE | SOLUSDT_1d | acc=0.5554


IN-SAMPLE | SOLBTC_1d | acc=0.6287


In [6]:
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

np.random.seed(100)
tf.random.set_seed(100)

def create_model(problem='regression'):
    model = Sequential()
    model.add(Dense(512, input_dim=len(cols),
                    activation='relu'))
    if problem == 'regression':
        model.add(Dense(1, activation='linear'))
        model.compile(loss='mse', optimizer='adam')
    else:
        model.add(Dense(1, activation='sigmoid'))
        model.compile(loss='binary_crossentropy', optimizer='adam')
    return model

for ric in data:
    model = create_model('classification')
    df, cols = dfs[ric]
    df[cols] = (df[cols] - df[cols].mean()) / df[cols].std()
    model.fit(df[cols], df['d'], epochs=50, verbose=False)
    pred = np.where(model.predict(df[cols]) > 0.5, 1, 0)
    acc = accuracy_score(df['d'], pred)
    print(f'IN-SAMPLE | {ric:7s} | acc={acc:.4f}')
    save_results('classification', ric, acc, "IN-SAMPLE")

C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 1/50 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


IN-SAMPLE | BTCUSDT_1d | acc=0.7274


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


...

IN-SAMPLE | SOLUSDT_1d | acc=0.6873


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 1/50 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


IN-SAMPLE | SOLBTC_1d | acc=0.6815


In [7]:
def walk_forward_fit_test(model):
    tscv = TimeSeriesSplit(n_splits=5, test_size=20)

    for ric in data:
        df, cols = dfs[ric]
        results = []
        for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
            train, test = df.iloc[train_idx], df.iloc[test_idx]

            X_train, y_train = train.drop(columns=['d']), train['d']
            X_test, y_test = test.drop(columns=['d']), test['d']

            # Normalizar usando solo datos de entrenamiento
            mean, std = X_train.mean(), X_train.std()
            X_train = (X_train - mean) / std
            X_test = (X_test - mean) / std  # Se usa la misma media y std de entrenamiento

            # Entrenar el modelo
            model.fit(X_train, y_train)

            # Predicción y evaluación
            pred = np.where(model.predict(X_test) > 0.5, 1, 0)
            acc = accuracy_score(y_test, pred)
            results.append(acc)

        # Guardar resultados
        avg_acc = np.mean(results)
        print(f'OUT-OF-SAMPLE | {ric:7s} | acc={acc:.4f}')
        save_results(type(model).__name__, ric, acc, "OUT-SAMPLE")

In [8]:
from sklearn.neural_network import MLPClassifier        

model_mlp = MLPClassifier(hidden_layer_sizes=[512],
                            random_state=100,
                            max_iter=1000,
                            early_stopping=True,
                            validation_fraction=0.15,
                            shuffle=False)

walk_forward_fit_test(model_mlp)

OUT-OF-SAMPLE | BTCUSDT_1d | acc=0.9000


OUT-OF-SAMPLE | LTCBTC_1d | acc=0.9500


OUT-OF-SAMPLE | BNBUSDT_1d | acc=0.9500


OUT-OF-SAMPLE | BNBBTC_1d | acc=1.0000


OUT-OF-SAMPLE | LTCUSDT_1d | acc=1.0000


OUT-OF-SAMPLE | ADABTC_1d | acc=0.9500


OUT-OF-SAMPLE | ADAUSDT_1d | acc=1.0000


OUT-OF-SAMPLE | SOLUSDT_1d | acc=1.0000


OUT-OF-SAMPLE | SOLBTC_1d | acc=1.0000


In [9]:
from sklearn.ensemble import BaggingClassifier

base_estimator = MLPClassifier(hidden_layer_sizes=[256],
                            random_state=100,
                            max_iter=1000,
                            early_stopping=True,
                            validation_fraction=0.15,
                            shuffle=False) 

model_bag = BaggingClassifier(base_estimator=base_estimator,
                            n_estimators=35,
                            max_samples=0.25,
                            max_features=0.5,
                            bootstrap=False,
                            bootstrap_features=True,
                            n_jobs=8,
                            random_state=100
                            )

walk_forward_fit_test(model_bag)

C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


...

C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


C:\Users\raque\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_base.py:156: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(


OUT-OF-SAMPLE | SOLBTC_1d | acc=0.9500
